# Logistic Regression Model

In [1]:
import pandas as pd # type: ignore
Data_final = pd.read_csv('/Users/instructorzamora/Documents/3_Maestria_Estadistica_UNINORTE/3_Tercer_Semestre/Machine_Learning/Deteccion_Fraude/Data_final.csv')
Data_final

,D12,D14,D11,D8,TransAmt,D3,D7,dist1,dist2,V209,...,V285,id_01,D13,isFraud,card4_discover,card4_mastercard,card4_visa,card6_credit,card6_debit,card6_debit or credit
0,0.0,0.0,13.0,37.875,68.500000,13.0,0.0,19.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,1,0,0,1,0,0
1,0.0,0.0,43.0,37.875,29.000000,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,1,0,0
2,0.0,0.0,315.0,37.875,59.000000,8.0,0.0,287.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,0,1,0,1,0
3,0.0,0.0,43.0,37.875,50.000000,0.0,0.0,8.0,37.0,0.0,...,10.0,-5.0,0.0,0.0,0,1,0,0,1,0
4,0.0,0.0,43.0,37.875,50.000000,8.0,0.0,8.0,37.0,0.0,...,0.0,0.0,0.0,0.0,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,0.0,0.0,56.0,37.875,49.000000,30.0,0.0,48.0,37.0,0.0,...,1.0,-5.0,0.0,0.0,0,0,1,0,1,0
590536,0.0,0.0,0.0,37.875,39.500000,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,0,1,0
590537,0.0,0.0,0.0,37.875,30.950001,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,0,1,0
590538,0.0,0.0,22.0,37.875,117.000000,0.0,0.0,3.0,37.0,0.0,...,5.0,-5.0,0.0,0.0,0,1,0,0,1,0


El dataset final es un conjunto de datos extenso de detección de fraude con 590,540 registros y 22 columnas, diseñado para un modelo de machine learning que busca identificar transacciones fraudulentas. Contiene variables numéricas como 'TransAmt' (monto de transacción), 'dist1', 'dist2', y códigos como D12, D14, D11, junto con variables categóricas binarias que representan características de tarjetas de pago (como tipos de tarjetas Discover, Mastercard, Visa, y tipos de tarjetas de crédito/débito). La variable objetivo 'isFraud' es binaria (0 o 1), indicando si una transacción es fraudulenta, mientras que la mayoría de las otras variables son numéricas con muchos valores cercanos a cero, sugiriendo un preprocesamiento de datos previo. Este dataset parece estar preparado para entrenar un modelo de clasificación que pueda predecir la probabilidad de fraude en transacciones financieras.

## Metricas Logistic Regression Model

In [6]:
# ------------------------
# Paso 1: Importar los paquetes necesarios
# ------------------------
import numpy as np # type: ignore
import pandas as pd # type: ignore
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from joblib import dump
from time import time
from skopt import BayesSearchCV
from skopt.space import Real, Categorical

# ------------------------
# Paso 2: Cargar los datos
# ------------------------
y = Data_final['isFraud']
X = Data_final.drop(columns=['isFraud'])

# ------------------------
# (Nuevo) Separar en entrenamiento y prueba
# ------------------------
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ------------------------
# Paso 3: Definir el pipeline
# ------------------------
pipe_logreg = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(solver='saga', max_iter=10000))  # solver saga permite elasticnet
])

# ------------------------
# Paso 4: Definir espacio de búsqueda para BayesSearchCV
# ------------------------
search_spaces = [
    ({'logreg__penalty': Categorical(['l1']),
      'logreg__C': Real(1e-4, 10.0, prior='log-uniform')}, 10),
    
    ({'logreg__penalty': Categorical(['l2']),
      'logreg__C': Real(1e-4, 10.0, prior='log-uniform')}, 10),
    
    ({'logreg__penalty': Categorical(['elasticnet']),
      'logreg__C': Real(1e-4, 10.0, prior='log-uniform'),
      'logreg__l1_ratio': Real(0.0, 1.0)}, 10)
]

# ------------------------
# Paso 5: Entrenar el modelo con BayesSearchCV
# ------------------------
bayes_logreg = BayesSearchCV(
    estimator=pipe_logreg,
    search_spaces=search_spaces,
    n_iter=30,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    random_state=42
)

start_time = time()
bayes_logreg.fit(x_train, y_train)
training_time_logreg = time() - start_time

# Guardar el modelo
dump(bayes_logreg, 'bayes_logreg.joblib')

# ------------------------
# Paso 6: Hacer predicciones
# ------------------------
y_pred_logreg = bayes_logreg.best_estimator_.predict(x_test)
y_pred_proba_logreg = bayes_logreg.best_estimator_.predict_proba(x_test)[:, 1]

# ------------------------
# Paso 7: Calcular métricas
# ------------------------
precision_logreg = precision_score(y_test, y_pred_logreg, average='weighted')
recall_logreg = recall_score(y_test, y_pred_logreg, average='weighted')
accuracy_logreg = accuracy_score(y_test, y_pred_logreg)
f1_logreg = f1_score(y_test, y_pred_logreg, average='weighted')
auc_logreg = roc_auc_score(y_test, y_pred_proba_logreg)

# ------------------------
# Paso 8: Resultados en DataFrame
# ------------------------
resultados_logreg = pd.DataFrame({
    'Precision': [f"{precision_logreg:.2f}"],
    'Recall': [f"{recall_logreg:.2f}"],
    'Accuracy': [f"{accuracy_logreg:.2f}"],
    'F1-Score': [f"{f1_logreg:.2f}"],
    'AUC': [f"{auc_logreg:.2f}"],
    'CPU time (s)': [round(training_time_logreg, 2)]
})

# ------------------------
# Paso 9: Mostrar resultados
# ------------------------
print("Métricas para el modelo Logistic Regression (Bayesian Optimization):")
display(resultados_logreg)  # Reemplaza por print(resultados_logreg) si no usas Jupyter


Métricas para el modelo Logistic Regression (Bayesian Optimization):


,Precision,Recall,Accuracy,F1-Score,AUC,CPU time (s)
0,0.95,0.96,0.96,0.95,0.73,20257.41


Para el modelo de Regresión Logística, el análisis revela un rendimiento robusto en la detección de fraudes. La precisión del 95% indica que cuando el modelo predice una transacción como fraudulenta, está en lo correcto en la mayoría de los casos. El recall del 96% sugiere que el modelo captura casi todas las transacciones fraudulentas reales. La accuracy del 96% confirma su alta efectividad general en la clasificación de transacciones. El F1-Score de 0.95 representa un excelente equilibrio entre precisión y recall. El AUC de 0.73, aunque aceptable, sugiere cierta dificultad en la separación perfecta de clases, lo cual es común en problemas de detección de fraude complejos. El tiempo de CPU de 20,257.41 segundos (aproximadamente 5.6 horas) refleja la complejidad computacional del modelo, posiblemente debido al extenso conjunto de datos de 590,540 registros que requiere un procesamiento significativo para ajustar los coeficientes de la regresión logística.